# 06 - Regression Metrics & Regularization

Notebook 01 used `Ridge` without explaining what the "ridge" is. This notebook fixes that.

By the end you will be able to:
1. Choose between MAE, RMSE and R^2 and say what each one punishes.
2. Explain what regularization actually does to the coefficients.
3. Use `Ridge` (L2) and `Lasso` (L1), and know when each is right.
4. Use Lasso to drop useless features automatically.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

cars = pd.read_csv("../data/car-sales-extended-missing-data.csv").dropna(subset=["Price"])
X = cars.drop("Price", axis=1)
y = cars["Price"]
print(X.shape)
cars.head()

## 1. The three regression metrics

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

categorical, numeric = ["Make", "Colour"], ["Odometer (KM)", "Doors"]

prep = ColumnTransformer([
    ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("e", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ("num", Pipeline([("i", SimpleImputer(strategy="median")),
                      ("s", StandardScaler())]), numeric),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lin = Pipeline([("prep", prep), ("model", LinearRegression())]).fit(X_train, y_train)
pred = lin.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"MAE  : {mae:9,.0f}   average error, in dollars")
print(f"RMSE : {rmse:9,.0f}   same units, but punishes big misses harder")
print(f"R^2  : {r2:9.3f}   fraction of variance explained (1.0 = perfect, 0 = no better than the mean)")

### Choosing between them

| Metric | Units | Behaviour | Use when |
|---|---|---|---|
| **MAE** | same as target | treats all errors equally | outliers are noise you want ignored |
| **RMSE** | same as target | squares errors, so one huge miss dominates | a large error is disproportionately costly |
| **R^2** | unitless | 1.0 perfect, 0 = predicting the mean, can go negative | comparing across different datasets |

**Report MAE to a stakeholder** - "we are off by about $4,000 on average" is a sentence
anyone understands. R^2 is for you, not for them.

In [ ]:
# Why RMSE > MAE always: one big miss.
errors = np.abs(y_test - pred)
plt.figure(figsize=(9, 4))
plt.hist(errors, bins=40, color="#4C78A8")
plt.axvline(mae, color="#F58518", ls="--", lw=2, label=f"MAE {mae:,.0f}")
plt.axvline(rmse, color="#E45756", ls="--", lw=2, label=f"RMSE {rmse:,.0f}")
plt.xlabel("absolute error ($)"); plt.ylabel("count")
plt.title("RMSE sits to the right of MAE - the tail is pulling it")
plt.legend()
plt.show()

## 2. What regularization actually does

Regularization adds a penalty on the **size of the coefficients** to the loss function.
The model must now trade fitting the data against keeping its weights small.

- **Ridge (L2)** penalises the sum of *squared* coefficients. It shrinks everything
  towards zero, but never exactly to zero.
- **Lasso (L1)** penalises the sum of *absolute* coefficients. It can push a coefficient
  to exactly zero, which removes the feature.

`alpha` controls the strength. `alpha=0` is plain linear regression. Large `alpha` means
a simpler, flatter model.

Let us watch it happen on data designed to overfit.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import make_pipeline

rng = np.random.RandomState(0)
xs = np.sort(rng.rand(25)) * 6
ys = np.sin(xs) + rng.randn(25) * 0.3
grid = np.linspace(0, 6, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, alpha in zip(axes, [0, 0.01, 1.0, 100.0]):
    model = (make_pipeline(PolynomialFeatures(15), LinearRegression()) if alpha == 0
             else make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=alpha)))
    model.fit(xs.reshape(-1, 1), ys)
    ax.scatter(xs, ys, s=22, color="#333333", zorder=3)
    ax.plot(grid, model.predict(grid), color="#E45756", lw=2)
    ax.plot(grid, np.sin(grid.ravel()), "--", color="grey", lw=1)
    ax.set_ylim(-2.2, 2.2)
    ax.set_title("no penalty" if alpha == 0 else f"Ridge alpha={alpha}")
plt.suptitle("Degree-15 polynomial, increasing regularization ->")
plt.tight_layout()
plt.show()

Left: wild swings, the classic overfit. Right: over-regularised, almost a flat line -
we have regularised the signal away too. The second and third panels are the useful range.

**`alpha` is a hyperparameter. You tune it with cross-validation (notebook 08), never by eye.**

## 3. Ridge vs Lasso on real coefficients

In [ ]:
# Build a dataset where only 5 of 40 features actually matter.
from sklearn.datasets import make_regression

Xr, yr, true_coef = make_regression(
    n_samples=200, n_features=40, n_informative=5,
    noise=15, coef=True, random_state=42)

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.3, random_state=42)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge (alpha=10)": Ridge(alpha=10),
    "Lasso (alpha=5)": Lasso(alpha=5, max_iter=10000),
}

fitted = {}
for name, m in models.items():
    m.fit(Xr_tr, yr_tr)
    fitted[name] = m
    print(f"{name:20} test R^2 = {m.score(Xr_te, yr_te):.3f} | "
          f"non-zero coefficients = {(np.abs(m.coef_) > 1e-6).sum():2d} / 40")
print(f"{'TRUTH':20} non-zero coefficients = {(true_coef != 0).sum():2d} / 40")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, (name, m) in zip(axes, fitted.items()):
    ax.bar(range(40), m.coef_, color="#4C78A8")
    ax.scatter(range(40), true_coef, color="#E45756", s=14, zorder=3, label="true coefficient")
    ax.set_ylabel(name.split(" ")[0]); ax.set_title(name, fontsize=10); ax.legend(fontsize=8)
axes[-1].set_xlabel("feature index")
plt.suptitle("Lasso zeroes out the 35 useless features; Ridge only shrinks them")
plt.tight_layout()
plt.show()

### The choice

| | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Coefficients | shrunk, never zero | many become exactly zero |
| Keeps all features? | yes | no - it selects |
| Correlated features | shares weight between them | picks one arbitrarily |
| Use when | all features plausibly matter | you suspect most are useless |

`ElasticNet` mixes both and is a reasonable default when you cannot decide.

**Always scale before regularising.** The penalty is on coefficient size, and a feature
measured in millions gets an artificially tiny coefficient - so it escapes the penalty.

---
## Exercises

1. Compute MAE, RMSE and R^2 for a `RandomForestRegressor` on the car data. Which metric
   shows the biggest improvement over linear regression, and why?
2. Add one artificial outlier row to the car data with `Price = 500000`. Re-fit and watch
   MAE and RMSE. Which one moved more? Which model would you trust now?
3. Sweep `alpha` for Ridge over `np.logspace(-3, 3, 20)` on the `make_regression` data and
   plot test R^2. Where is the peak?
4. Do the same sweep for Lasso and plot the number of non-zero coefficients against alpha.
5. **Stretch:** use `RidgeCV` and `LassoCV`, which tune alpha internally by cross-validation.
   Compare their chosen alpha to your manual sweep.